# Experiment 3 -- PhotonFlow + Noise Regularization on MNIST

**Purpose:** Test noise-aware training — does injecting photonic noise during training improve hardware robustness?

| Noise Source | σ | Physics | Hardware |
|---|---|---|---|
| **Shot noise** | σ_s = 0.02 | Poisson photon counting → additive iid Gaussian | Photodetector readout |
| **Thermal crosstalk** | σ_t = 0.01 | Heat diffusion between adjacent phase shifters → correlated Gaussian | MZI thermo-optic tuner |

**Key question:** Does training with noise injection improve FID compared to exp2 (no noise)?

**Training improvements:** Curriculum sampling, EMA, cosine LR, velocity direction loss, time-dependent loss weighting.

**References:** Shen 2017, Ning 2025, Dao 2022, Lipman 2023, Esser 2024, Karras 2024.


In [ ]:
# -- 1. Environment detection + dependency install -------------------------
import sys, subprocess, os

# Detect environment (check Kaggle FIRST -- Kaggle can also import google.colab)
IN_COLAB  = False
IN_KAGGLE = False
IN_LOCAL  = False

if os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    IN_KAGGLE = True
    ENV_NAME  = "Kaggle"
else:
    try:
        import google.colab
        IN_COLAB = True
        ENV_NAME = "Colab"
    except ImportError:
        IN_LOCAL = True
        ENV_NAME = "Local"

print(f"Environment: {ENV_NAME}")

# ---- Install dependencies (resilient: per-package, warn on failure) ----
# exp2 does NOT need torchcfm or photontorch -- we use our own
# photonflow.train.CFMLoss and photonflow.model.PhotonFlowModel.
# torchcfm/photontorch are only needed for exp6 (hardware simulation).

def _pip_install(*pkgs):
    """Install packages one by one. Warn on failure instead of crashing."""
    for pkg in pkgs:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            print(f"  [ok] {pkg}")
        except subprocess.CalledProcessError:
            print(f"  [warn] {pkg} -- install failed (may need Internet)")
            if IN_KAGGLE:
                print("         Kaggle: Settings (right panel) > Internet > toggle ON")

if IN_COLAB or IN_KAGGLE:
    print("Installing dependencies...")
    _pip_install("tqdm", "pyyaml", "wandb")
    # Optional: uncomment if this experiment needs torchonn
    # _pip_install("git+https://github.com/HasinthakaPiyumal/pytorch-onn.git")

import torch
assert torch.cuda.is_available(), (
    "GPU not found.\n"
    "  Colab : Runtime > Change runtime type > T4 GPU\n"
    "  Kaggle: Settings > Accelerator > GPU T4 x2"
)
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# -- 2. Repo / sys.path setup ----------------------------------------------
REPO_URL    = "https://github.com/HasinthakaPiyumal/photon-flow-research.git"
REPO_BRANCH = "h/phase1"  # <-- change this to switch branches

if IN_COLAB:
    REPO_DIR = "/content/photon-flow-research"
elif IN_KAGGLE:
    REPO_DIR = "/kaggle/working/photon-flow-research"
else:
    REPO_DIR = ".."

if (IN_COLAB or IN_KAGGLE) and not os.path.exists(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR
    ])
    print(f"Cloned repo (branch={REPO_BRANCH}) to {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo  : {os.path.abspath(REPO_DIR)}")
print(f"Branch: {REPO_BRANCH}")

In [ ]:
# -- 3. Imports -------------------------------------------------------------
import math, json
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# PhotonFlow core
from photonflow.model import PhotonFlowModel       # Dao 2022 Monarch architecture
from photonflow.train import CFMLoss, euler_sample, EMA  # + EMA (Karras 2024)

# FID evaluation (Heusel 2017)
from eval.fid import FIDCalculator

device = torch.device("cuda")
print(f"Device: {device}")
print("Imports OK")


In [ ]:
# -- W&B init (get API key from secrets) -----------------------------------
import wandb

# Retrieve WANDB_API_KEY from environment secrets
# Colab : Add-ons > Secrets > "WANDB_API_KEY"
# Kaggle: Add-ons > Secrets > "WANDB_API_KEY" (toggle Internet ON)
# Local : export WANDB_API_KEY=... in shell
WANDB_API_KEY = None

if IN_COLAB:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print(f"[warn] Could not load WANDB_API_KEY from Colab secrets: {e}")

elif IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception as e:
        print(f"[warn] Could not load WANDB_API_KEY from Kaggle secrets: {e}")

else:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("W&B login: OK")
else:
    print("[warn] WANDB_API_KEY not found -- running wandb in offline mode")
    os.environ["WANDB_MODE"] = "offline"

wandb_run = None
print("W&B cell loaded -- run will be initialised after config is ready")

In [ ]:
# -- 4. Experiment config + W&B run init -----------------------------------
import yaml

config_path = os.path.join(REPO_DIR, "configs", "exp3_noise.yaml")
with open(config_path) as f:
    yaml_cfg = yaml.safe_load(f)

print(f"Loaded config: {config_path}")

CFG = {
    "in_dim"     : yaml_cfg["model"]["in_dim"],
    "hidden_dim" : yaml_cfg["model"]["hidden_dim"],
    "num_blocks" : yaml_cfg["model"]["num_blocks"],
    "use_noise"  : yaml_cfg["model"]["use_noise"],
    "sigma_s"    : yaml_cfg["model"]["sigma_s"],
    "sigma_t"    : yaml_cfg["model"]["sigma_t"],
    "noise_warmup_steps": yaml_cfg["model"].get("noise_warmup_steps", 0),
    "gate_init"  : yaml_cfg["model"].get("gate_init", 0.0),
    "depth_decay_residual": yaml_cfg["model"].get("depth_decay_residual", False),
    "adaln_init_std": yaml_cfg["model"].get("adaln_init_std", 0.0),
    "num_monarch_factors": yaml_cfg["model"].get("num_monarch_factors", 1),
    "dataset"    : yaml_cfg["data"]["dataset"],
    "batch_size" : yaml_cfg["data"]["batch_size"],
    "data_root"  : os.path.join(REPO_DIR, yaml_cfg["data"]["root"]),
    "lr"               : yaml_cfg["training"]["lr"],
    "total_steps"      : yaml_cfg["training"]["total_steps"],
    "checkpoint_every" : yaml_cfg["training"]["checkpoint_every"],
    "sample_every"     : yaml_cfg["training"]["sample_every"],
    "sample_steps"     : yaml_cfg["training"]["sample_steps"],
    "seed"             : yaml_cfg["training"]["seed"],
    "warmup_steps"     : yaml_cfg["training"].get("warmup_steps", 0),
    "lr_schedule"      : yaml_cfg["training"].get("lr_schedule", "constant"),
    "time_sampling"    : yaml_cfg["training"].get("time_sampling", "uniform"),
    "logit_normal_mean": yaml_cfg["training"].get("logit_normal_mean", 0.0),
    "logit_normal_std" : yaml_cfg["training"].get("logit_normal_std", 1.0),
    "curriculum_transition_frac": yaml_cfg["training"].get("curriculum_transition_frac", 0.6),
    "direction_loss_weight": yaml_cfg["training"].get("direction_loss_weight", 0.0),
    "loss_weight_gamma": yaml_cfg["training"].get("loss_weight_gamma", 0.0),
    "ema_enabled" : yaml_cfg.get("ema", {}).get("enabled", False),
    "ema_decay"   : yaml_cfg.get("ema", {}).get("decay", 0.9999),
    "output_dir"  : os.path.join(REPO_DIR, yaml_cfg["output_dir"]),
}
CFG["time_dim"] = yaml_cfg["model"].get("time_dim", 256)

IN_DIM = CFG["in_dim"]
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

OUTPUT_DIR = CFG["output_dir"]
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
for d in [CKPT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

_ts_k = CFG["total_steps"] // 1000
wandb_run = wandb.init(
    project="photonflow",
    name=f"exp3-noise-mnist-{_ts_k}K",
    config=CFG,
    tags=["exp3", "photonflow", "noise", "monarch", "mnist", ENV_NAME.lower()],
    notes=yaml_cfg["experiment"]["description"],
    reinit=True,
)

print(f"Model: PhotonFlowModel, hidden={CFG['hidden_dim']}, blocks={CFG['num_blocks']}")
print(f"  Noise: sigma_s={CFG['sigma_s']}, sigma_t={CFG['sigma_t']}")
print(f"  Noise warmup: {CFG['noise_warmup_steps']} steps")
print(f"  EMA: {CFG['ema_enabled']}, decay={CFG['ema_decay']}")
print(f"  Time sampling: {CFG['time_sampling']}")
print(f"Training: {CFG['total_steps']:,} steps, bs={CFG['batch_size']}, lr={CFG['lr']}")
print(f"W&B run: {wandb_run.url}")


In [ ]:
# -- 5. MNIST DataLoader ---------------------------------------------------
_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda img: img.view(-1)),
])

train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=_tfm)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=_tfm)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)

steps_per_epoch = len(train_loader)
total_epochs    = CFG["total_steps"] / steps_per_epoch
_bs = CFG["batch_size"]; _ts = CFG["total_steps"]
print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print(f"Batch size: {_bs}  Steps/epoch: {steps_per_epoch}")
print(f"Total epochs: {total_epochs:.1f}  ({_ts:,} steps)")

In [ ]:
# -- 6. PhotonFlowModel + EMA + advanced CFMLoss --------------------------
model = PhotonFlowModel(
    in_dim     = CFG["in_dim"],
    hidden_dim = CFG["hidden_dim"],
    num_blocks = CFG["num_blocks"],
    time_dim   = CFG["time_dim"],
    use_noise  = CFG["use_noise"],       # True -- noise injection ON
    sigma_s    = CFG["sigma_s"],          # 0.02 -- shot noise
    sigma_t    = CFG["sigma_t"],          # 0.01 -- thermal crosstalk
    gate_init  = CFG.get("gate_init", 0.0),
    depth_decay_residual = CFG.get("depth_decay_residual", False),
    adaln_init_std = CFG.get("adaln_init_std", 0.0),
    num_monarch_factors = CFG.get("num_monarch_factors", 1),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])

# Advanced CFMLoss with all training improvements
loss_fn = CFMLoss(
    sigma_min=0.0,
    time_sampling=CFG["time_sampling"],
    logit_normal_mean=CFG["logit_normal_mean"],
    logit_normal_std=CFG["logit_normal_std"],
    curriculum_transition_frac=CFG["curriculum_transition_frac"],
    direction_loss_weight=CFG["direction_loss_weight"],
    loss_weight_gamma=CFG["loss_weight_gamma"],
)

# EMA (Karras et al. 2024)
ema = EMA(model, decay=CFG["ema_decay"]) if CFG["ema_enabled"] else None
print(f"EMA: {'enabled' if ema else 'disabled'}")

# LR scheduler
from torch.optim.lr_scheduler import LambdaLR

warmup_steps = CFG.get("warmup_steps", 0)
lr_schedule  = CFG.get("lr_schedule", "constant")

if lr_schedule == "cosine":
    _total = CFG["total_steps"]
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(_total - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = LambdaLR(optimizer, lr_lambda)
else:
    scheduler = None

total_params = model.count_parameters()
print(f"Model: PhotonFlowModel")
print(f"  hidden_dim = {CFG['hidden_dim']}, blocks = {CFG['num_blocks']}")
print(f"  use_noise = {CFG['use_noise']}, sigma_s = {CFG['sigma_s']}, sigma_t = {CFG['sigma_t']}")
print(f"  num_monarch_factors = {CFG['num_monarch_factors']}")
print(f"  params = {total_params:,}")

wandb.log({"model/total_params": total_params})

with torch.no_grad():
    _x = torch.randn(4, CFG["in_dim"], device=device)
    _t = torch.rand(4, device=device)
    _max = model(_x, _t).abs().max().item()
print(f"Init output max abs: {_max:.6f}")
print("Initialization OK")


In [ ]:
# -- 7. VERIFY FIRST 1K STEPS (quick sanity check) -------------------------
VERIFY_STEPS = 1000

verify_losses = []
data_iter_v = iter(train_loader)
model.train()

print(f"Verification run: {VERIFY_STEPS} steps (with noise injection)...")
print()

for step in range(VERIFY_STEPS):
    try:
        x1, _ = next(data_iter_v)
    except StopIteration:
        data_iter_v = iter(train_loader)
        x1, _ = next(data_iter_v)

    x1 = x1.to(device)
    # Noise warmup: ramp noise from 0 to full over noise_warmup_steps
    nw = CFG["noise_warmup_steps"]
    if nw > 0:
        model.set_noise_scale(min(1.0, step / nw))
    loss = loss_fn(model, x1, step=step, total_steps=CFG["total_steps"])

    optimizer.zero_grad()
    loss.backward()

    if step == 0:
        no_grad = [n for n, p in model.named_parameters() if p.grad is None]
        assert len(no_grad) == 0, f"Params with no gradient: {no_grad}"

    optimizer.step()
    if scheduler is not None:
        scheduler.step()
    if ema is not None:
        ema.update(model)
    verify_losses.append(loss.item())

    if (step + 1) % 200 == 0:
        avg = np.mean(verify_losses[-200:])
        print(f"  step {step+1:>5d}/{VERIFY_STEPS}  loss={avg:.4f}")

first_100 = np.mean(verify_losses[:100])
last_100  = np.mean(verify_losses[-100:])
any_nan   = any(math.isnan(l) for l in verify_losses)
decreased = last_100 < first_100

print()
print(f"  Loss first 100: {first_100:.4f}")
print(f"  Loss last  100: {last_100:.4f}")
print(f"  Decreased:      {decreased}")
print(f"  Any NaN:        {any_nan}")

# Sample using EMA weights if available
model.eval()
if ema is not None:
    ema_backup = ema.apply(model)
with torch.no_grad():
    quick_samp = euler_sample(model, shape=(16, IN_DIM),
                              num_steps=CFG["sample_steps"], device=str(device))
if ema is not None:
    ema.restore(model, ema_backup)

quick_samp = quick_samp.clamp(0.0, 1.0).cpu().view(16, 1, 28, 28)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(quick_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(f"Exp3 verification (1K steps, loss={last_100:.4f})", fontsize=11)
plt.tight_layout()
wandb.log({"verify/samples_1K": wandb.Image(fig)})
plt.show()
plt.close(fig)

model.train()

if decreased and not any_nan:
    print("\n" + "=" * 50)
    print("  VERDICT: GO -- proceed to full training")
    print("=" * 50)
else:
    print("\n" + "=" * 50)
    print("  VERDICT: NO-GO -- debug before full training")
    print("=" * 50)

# Reset for full training
model = PhotonFlowModel(
    in_dim=CFG["in_dim"], hidden_dim=CFG["hidden_dim"],
    num_blocks=CFG["num_blocks"], time_dim=CFG["time_dim"],
    use_noise=CFG["use_noise"], sigma_s=CFG["sigma_s"], sigma_t=CFG["sigma_t"],
    gate_init=CFG.get("gate_init", 0.0),
    depth_decay_residual=CFG.get("depth_decay_residual", False),
    adaln_init_std=CFG.get("adaln_init_std", 0.0),
    num_monarch_factors=CFG.get("num_monarch_factors", 1),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
if lr_schedule == "cosine":
    scheduler = LambdaLR(optimizer, lr_lambda)
else:
    scheduler = None
ema = EMA(model, decay=CFG["ema_decay"]) if CFG["ema_enabled"] else None
torch.manual_seed(CFG["seed"])
print("\nModel reset for full training.")


In [ ]:
# -- 8. Training loop (200K steps) -----------------------------------------
import wandb
losses   = []
step_log = []

data_iter = iter(train_loader)
model.train()

pbar = tqdm(range(CFG["total_steps"]), desc="exp3 Noise", dynamic_ncols=True)

for step in pbar:

    try:
        x1, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x1, _ = next(data_iter)

    x1 = x1.to(device)

    # Noise warmup: ramp noise 0 -> full over noise_warmup_steps
    nw = CFG["noise_warmup_steps"]
    if nw > 0:
        model.set_noise_scale(min(1.0, step / nw))

    loss = loss_fn(model, x1, step=step, total_steps=CFG["total_steps"])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if scheduler is not None:
        scheduler.step()
    if ema is not None:
        ema.update(model)

    losses.append(loss.item())
    wandb.log({"train/loss": loss.item()}, step=step)

    if step % 100 == 0:
        avg = float(np.mean(losses[max(0, len(losses)-100):]))
        step_log.append((step, avg))
        pbar.set_postfix(loss=f"{avg:.4f}")
        wandb.log({"train/loss_avg100": avg}, step=step)

    if step % 1000 == 0:
        cur_lr = optimizer.param_groups[0]["lr"]
        noise_s = min(1.0, step / max(CFG["noise_warmup_steps"], 1))
        wandb.log({"train/lr": cur_lr, "train/noise_scale": noise_s}, step=step)

    # Checkpoint every 5K
    if (step + 1) % CFG["checkpoint_every"] == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"step_{step+1:07d}.pt")
        save_dict = {
            "step": step + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "losses": losses,
            "config": CFG,
        }
        if ema is not None:
            save_dict["ema_state_dict"] = ema.state_dict()
        torch.save(save_dict, ckpt_path)
        art = wandb.Artifact(f"exp3-checkpoint-step{step+1}", type="model")
        art.add_file(ckpt_path)
        wandb.log_artifact(art)
        print(f"\n[step {step+1:,}] Checkpoint -> {ckpt_path}")

    # Sample grid every 5K (using EMA weights)
    if (step + 1) % CFG["sample_every"] == 0:
        model.eval()
        if ema is not None:
            ema_backup = ema.apply(model)
        with torch.no_grad():
            samp = euler_sample(model, shape=(64, IN_DIM),
                                num_steps=CFG["sample_steps"], device=str(device))
        if ema is not None:
            ema.restore(model, ema_backup)
        samp = samp.clamp(0.0, 1.0).cpu().view(64, 1, 28, 28)

        fig, axes = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            ax.imshow(samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
        avg_now = float(np.mean(losses[max(0, len(losses)-500):]))
        fig.suptitle(f"Exp3 Noise  step={step+1:,}  loss={avg_now:.4f}", fontsize=11)
        plt.tight_layout()
        fig_path = os.path.join(FIG_DIR, f"samples_step{step+1:07d}.png")
        plt.savefig(fig_path, dpi=100, bbox_inches="tight")
        wandb.log({f"samples/step_{step+1}": wandb.Image(fig)}, step=step)
        plt.show()
        plt.close(fig)
        model.train()

pbar.close()
print(f"\nTraining complete: {step+1:,} steps, final loss: {losses[-1]:.4f}")


In [ ]:
# -- 9. Loss curve ---------------------------------------------------------
import wandb
steps_arr  = [s for s, _ in step_log]
losses_arr = [l for _, l in step_log]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(steps_arr, losses_arr, lw=1.5, alpha=0.85, color="crimson")
ax.set_xlabel("Training step", fontsize=12)
ax.set_ylabel("CFM loss (MSE)", fontsize=12)
_ts_k = CFG["total_steps"] // 1000
ax.set_title(f"Exp3: PhotonFlow + Noise on MNIST ({_ts_k}K steps)", fontsize=13)
ax.grid(True, alpha=0.3)
ax.annotate(
    f"final avg = {np.mean(losses[-1000:]):.4f}",
    xy=(steps_arr[-1], losses_arr[-1]),
    xytext=(-120, 10), textcoords="offset points",
    fontsize=10, color="crimson")
plt.tight_layout()
curve_path = os.path.join(FIG_DIR, "loss_curve.png")
plt.savefig(curve_path, dpi=150)
wandb.log({"charts/loss_curve": wandb.Image(fig, caption="Training loss curve")})
plt.show()
plt.close(fig)
print(f"Loss curve saved: {curve_path}")


In [ ]:
# -- 10. Final sample grid (10x10, EMA weights) ----------------------------
import wandb
model.eval()
if ema is not None:
    ema_backup = ema.apply(model)
with torch.no_grad():
    final_samp = euler_sample(model, shape=(100, IN_DIM),
                              num_steps=CFG["sample_steps"], device=str(device))
if ema is not None:
    ema.restore(model, ema_backup)

final_samp = final_samp.clamp(0.0, 1.0).cpu().view(100, 1, 28, 28)

fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(final_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
_ts = CFG["total_steps"]
fig.suptitle(f"Exp3 Final Samples (step {_ts:,}, EMA={CFG['ema_enabled']})", fontsize=13)
plt.tight_layout()
samp_path = os.path.join(FIG_DIR, "final_samples.png")
plt.savefig(samp_path, dpi=150, bbox_inches="tight")
wandb.log({"samples/final_10x10": wandb.Image(fig, caption="Final 100 generated samples (EMA)")})
plt.show()
plt.close(fig)
print(f"Final samples saved: {samp_path}")


In [ ]:
# -- 11. FID computation ---------------------------------------------------
import wandb
N_FID    = 10_000
fid_calc = FIDCalculator(device=device)

real_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
real_imgs = []
for imgs, _ in real_loader:
    real_imgs.append(imgs.view(-1, 1, 28, 28))
real_imgs = torch.cat(real_imgs, dim=0)[:N_FID]
print(f"Real images: {real_imgs.shape}")

# Generate using EMA weights
model.eval()
if ema is not None:
    ema_backup = ema.apply(model)

gen_batches = []
GEN_BATCH   = 256
n_generated = 0
with torch.no_grad():
    pbar_fid = tqdm(total=N_FID, desc="Generating for FID", unit="img")
    while n_generated < N_FID:
        bs   = min(GEN_BATCH, N_FID - n_generated)
        samp = euler_sample(model, shape=(bs, IN_DIM),
                            num_steps=CFG["sample_steps"], device=str(device))
        gen_batches.append(samp.clamp(0.0, 1.0).cpu().view(bs, 1, 28, 28))
        n_generated += bs
        pbar_fid.update(bs)
    pbar_fid.close()

if ema is not None:
    ema.restore(model, ema_backup)

gen_imgs = torch.cat(gen_batches, dim=0)[:N_FID]

print("Extracting real features...")
real_feats = fid_calc.extract_features(real_imgs, batch_size=256)
print("Extracting generated features...")
gen_feats  = fid_calc.extract_features(gen_imgs, batch_size=256)

real_stats = fid_calc.compute_statistics(real_feats)
gen_stats  = fid_calc.compute_statistics(gen_feats)
fid_score  = fid_calc.compute_fid(real_stats, gen_stats)

_ts_k = CFG["total_steps"] // 1000
print(f"\nFID (exp3 Noise, {_ts_k}K steps): {fid_score:.2f}")
wandb.log({"eval/fid": fid_score})

# Compare against exp2 (no noise) and exp1 (baseline)
for exp_name, exp_dir, exp_file in [
    ("exp2 (no noise)", "exp2_photonflow_mnist", "results_exp2.json"),
    ("exp1 (baseline)", "exp1_baseline", "results_exp1.json"),
]:
    prev_path = os.path.join(REPO_DIR, "outputs", exp_dir, exp_file)
    if os.path.exists(prev_path):
        with open(prev_path) as f:
            prev = json.load(f)
        prev_fid = prev["fid"]
        delta_pct = ((fid_score - prev_fid) / prev_fid) * 100
        better = "BETTER" if fid_score < prev_fid else "WORSE"
        print(f"  vs {exp_name}: FID {prev_fid:.2f} -> {fid_score:.2f} ({delta_pct:+.1f}%) [{better}]")
    else:
        print(f"  ({exp_name} results not found -- run that notebook first)")


In [ ]:
# -- 12. Results summary + W&B finish --------------------------------------
import json as _json, wandb

results = {
    "experiment"   : "exp3_noise_regularized",
    "environment"  : ENV_NAME,
    "description"  : f"PhotonFlow + noise (sigma_s={CFG['sigma_s']}, sigma_t={CFG['sigma_t']}) on MNIST",
    "total_steps"  : CFG["total_steps"],
    "fid"          : round(float(fid_score), 4),
    "final_loss_avg1000": round(float(np.mean(losses[-1000:])), 6),
    "model_params" : total_params,
    "architecture" : {
        "type"       : "photonflow",
        "hidden_dim" : CFG["hidden_dim"],
        "num_blocks" : CFG["num_blocks"],
        "time_dim"   : CFG["time_dim"],
        "use_noise"  : CFG["use_noise"],
        "sigma_s"    : CFG["sigma_s"],
        "sigma_t"    : CFG["sigma_t"],
        "ema_decay"  : CFG["ema_decay"],
    },
    "training" : {
        "optimizer"      : "Adam",
        "lr"             : CFG["lr"],
        "batch_size"     : CFG["batch_size"],
        "dataset"        : CFG["dataset"],
        "time_sampling"  : CFG["time_sampling"],
        "direction_loss" : CFG["direction_loss_weight"],
        "loss_weight_gamma": CFG["loss_weight_gamma"],
    },
    "sources" : [
        "Shen et al. 2017 -- shot noise + thermal crosstalk (Nature Photonics)",
        "Ning et al. 2025 -- noise-aware training (StrC-ONN)",
        "Dao et al. 2022 -- Monarch matrices (ICML)",
        "Lipman et al. 2023 -- CFM loss (ICLR)",
        "Esser et al. 2024 -- logit-normal sampling (SD3)",
        "Karras et al. 2024 -- EMA (CVPR)",
    ],
}

results_path = os.path.join(OUTPUT_DIR, "results_exp3.json")
with open(results_path, "w") as f:
    _json.dump(results, f, indent=2)

np.save(os.path.join(OUTPUT_DIR, "losses_exp3.npy"), np.array(losses))

final_ckpt = os.path.join(CKPT_DIR, "exp3_final.pt")
save_dict = {
    "step": CFG["total_steps"],
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fid": fid_score,
    "config": CFG,
}
if ema is not None:
    save_dict["ema_state_dict"] = ema.state_dict()
torch.save(save_dict, final_ckpt)

wandb.log({
    "summary/fid"             : fid_score,
    "summary/final_loss"      : np.mean(losses[-1000:]),
    "summary/total_params"    : total_params,
})
wandb.save(results_path)
wandb.finish()

m = math.isqrt(CFG["hidden_dim"])
SEP = "=" * 58
print(SEP)
print("EXP3 NOISE-REGULARIZED RESULTS")
print(SEP)
print(f"  FID (10K samples):      {fid_score:.2f}")
print(f"  Final CFM loss (1K):    {np.mean(losses[-1000:]):.4f}")
print(f"  Model parameters:       {total_params:,}")
print(f"  Training steps:         {CFG['total_steps']:,}")
print(f"  Architecture:           PhotonFlow ({CFG['num_blocks']} blocks, Monarch m={m})")
print(f"  Photonic noise:         sigma_s={CFG['sigma_s']}, sigma_t={CFG['sigma_t']}")
print(f"  EMA:                    {CFG['ema_enabled']} (decay={CFG['ema_decay']})")
print(f"  Environment:            {ENV_NAME}")
print(SEP)
print(f"  Results    -> {results_path}")
print(f"  Checkpoint -> {final_ckpt}")
print(f"  W&B run    -> {wandb_run.url}")
print(SEP)
print()
print("Next: notebooks/05_exp4_qat_finetune.ipynb")
print("      Load exp3 checkpoint -> 4-bit QAT fine-tune -> hardware-ready model")
